---
title: "3.2 Making a chart readable"
---

::: {.lesson-tools}
[Open in Colab](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/lessons/03-viz/02-readable-charts.ipynb){.btn .btn-primary}
[Download notebook](../../downloads/03-viz/02-readable-charts.ipynb){.btn .btn-outline-secondary}
:::

**Run in your browser.** Colab needs a Google account to run and save work. If its GitHub link cannot open, download this notebook and use **File → Upload notebook** at [Colab](https://colab.research.google.com/). You can read the website without an account. A notebook is a document containing explanatory text and runnable code cells.

**Time:** about 40 minutes, including exercises · **Before you begin:** Complete [3.1 First plots from a DataFrame](../03-viz/01-first-plots.ipynb) first. Each notebook starts afresh: it does not remember variables from another lesson.

## You will learn

- Build a scatter plot with clear axes, units and a specific observation year.
- Explain a logarithmic axis without changing the stored measurement.
- Use annotation and redundant visual cues instead of relying on colour alone.

## Why it matters

The running question compares income and life expectancy, two numeric measurements. A **scatter plot** places one point at each observation's pair of values. Labels, scales and a caption determine what a reader can learn from the picture. This lesson first compares made-up numbers for fictional countries, then recreates the prepared chart from Part 0 with every main choice explained.

## Worked example

### Put two measurements on two axes

The x-axis is horizontal and the y-axis vertical. Use one row per fictional country in one year; no connecting line is appropriate because these countries have no natural sequence. Labels include units rather than just code names.

In [ ]:
#| label: fig-fictional-scatter
#| fig-alt: "Three fictional country points show life expectancy rising from Southport at 4200 dollars and 64 years to Eastria at 30000 dollars and 80.1 years."
#| fig-cap: "Made-up numbers; three points are only a command demonstration."
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 12})
small = pd.DataFrame({"country": ["Northland", "Eastria", "Southport"], "gdp_pc": [15000, 30000, 4200], "life_exp": [73.5, 80.1, 64.0]})
fig_small, ax_small = plt.subplots(figsize=(6, 4), layout="constrained")
ax_small.scatter(small["gdp_pc"], small["life_exp"], color="#21645a", marker="o")
ax_small.set(xlabel="GDP per person (made-up dollars)", ylabel="Life expectancy (years)", title="Three fictional countries · made-up numbers")
plt.show()

The points describe paired measurements. Switching which variable is on the horizontal axis changes the presentation, but neither choice establishes causation.

### Prepare the real country panel

Run this prepared setup once in a fresh session. It uses the course CSV in a local checkout, or downloads its fixed, checksum-verified public source in Colab. A checksum is a fingerprint of a file's contents; a mismatch stops the download from being used. You do not need to retype the setup. It does not use your Google Drive or any private data.

In [ ]:
# Prepared setup: use the course CSV locally; fetch its pinned source in Colab.
from pathlib import Path
import hashlib
import io
from urllib.request import urlopen
import pandas as pd

data_file = Path("data/gapminder.csv")
if not data_file.exists():
    source = "https://raw.githubusercontent.com/jennybc/gapminder/391b5a40d3574c3b89d889dd7d72f7f3fdb4a70c/inst/extdata/gapminder.tsv"
    with urlopen(source, timeout=30) as response:
        raw = response.read()
    expected = "f74275b4a2f4a7c8e3a09a752f9d6e43d1759d78d5cbfdebaaa3e1a6efc68f78"
    if hashlib.sha256(raw).hexdigest() != expected:
        raise ValueError("The data download changed. Please report this to the course maintainer.")
    downloaded = pd.read_csv(io.BytesIO(raw), sep="\t", dtype=str)
    downloaded = downloaded.rename(columns={"continent": "region", "lifeExp": "life_exp", "pop": "population", "gdpPercap": "gdp_pc"})
    data_file.parent.mkdir(parents=True, exist_ok=True)
    downloaded.to_csv(data_file, index=False, lineterminator="\n")
print("Course data ready: data/gapminder.csv")

Look for **Course data ready**. Colab working files are temporary; rerun this cell after a runtime reset. The data cover 142 countries at five-year intervals from 1952 to 2007. Each row is one country in one year.

### Choose the sample and check log eligibility

Load the panel, select 2007 and count the rows and missing values. The GDP values must be positive to use a real logarithmic scale. A plot should not silently discard zero or negative measurements.

In [ ]:
countries = pd.read_csv(data_file)
year_2007 = countries.loc[countries["year"] == 2007].copy()
print("Countries:", len(year_2007))
print(year_2007[["gdp_pc", "life_exp"]].isna().sum())
print("All GDP values positive:", year_2007["gdp_pc"].gt(0).all())

There are 142 countries, no missing values in the plotted fields and all GDP-per-person values are positive. These checks support this display; a real raw dataset may need an explicit excluded-observation count.

### Compare a linear and a logarithmic axis

A linear axis uses equal spacing for equal dollar differences. A **logarithmic axis** uses equal spacing for equal ratios: for example, 1000 to 2000 has the same horizontal distance as 10000 to 20000. `set_xscale("log")` changes only the display, not the stored GDP column. Use two panels with the same vertical range to compare.

In [ ]:
#| label: fig-compare-scales
#| fig-alt: "Two scatter panels contain the same 142 country observations in 2007. A log income axis spreads low-income observations and compresses the high-income range compared with a linear axis; life-expectancy values are unchanged."
#| fig-cap: "Historical Gapminder teaching extract via Jennifer Bryan (CC0). Country observations receive equal weight unless stated otherwise."
fig_scales, axes_scales = plt.subplots(2, 1, figsize=(6, 8), sharey=True, layout="constrained")
for axis, scale in zip(axes_scales, ["linear", "log"]):
    axis.scatter(year_2007["gdp_pc"], year_2007["life_exp"], color="#21645a", alpha=0.8, edgecolor="white", linewidth=0.4)
    axis.set_xscale(scale)
    axis.set(xlabel=f"GDP per person (inflation-adjusted US dollars; {scale} axis)", ylabel="Life expectancy (years)", title=f"2007 · {scale} income axis")
    axis.grid(axis="y", alpha=0.25)
plt.show()

The log axis makes a broad upward association easier to see without changing its data. It is not the same thing as relabelling GDP in thousands. A reader must be told that the scale is logarithmic. A log-transformed regression in Part 4 changes the model's predictor, not merely an axis.

### Add one useful annotation

An **annotation** attaches text to a point. Select Canada's 2007 row, then offset a short label in points (a display unit) so it does not cover the marker. A uniform dark marker and horizontal grid supply structure without a colour legend.

In [ ]:
#| label: fig-annotated-canada
#| fig-alt: "Scatter plot of 142 countries in 2007 with a log income axis and a labelled Canada point at about 36319 dollars per person and 80.653 years life expectancy."
#| fig-cap: "Historical Gapminder teaching extract via Jennifer Bryan (CC0). Country observations receive equal weight unless stated otherwise. The chart describes an association, not a causal effect."
canada = year_2007.loc[year_2007["country"] == "Canada"].iloc[0]
fig_readable, ax_readable = plt.subplots(figsize=(6, 4.5), layout="constrained")
ax_readable.scatter(year_2007["gdp_pc"], year_2007["life_exp"], color="#21645a", alpha=0.8, edgecolor="white", linewidth=0.4)
ax_readable.set_xscale("log")
ax_readable.set(xlabel="GDP per person (inflation-adjusted US dollars; log axis)", ylabel="Life expectancy at birth (years)", title="Income and life expectancy · 2007")
ax_readable.annotate("Canada", xy=(canada["gdp_pc"], canada["life_exp"]), xytext=(-48, -22), textcoords="offset points", arrowprops={"arrowstyle": "-", "color": "black"})
ax_readable.grid(axis="y", alpha=0.25)
plt.show()

The annotation identifies one observation; it does not imply that it is representative. The other points retain equal size, so no population weighting is encoded. Avoid labelling every country if the text overlaps; provide a table when exact identification matters.

## Try it

Work in your own notebook copy. Add a new code cell for each task, try it yourself, then open the solution to compare. Run from the top if a name is missing.

### 1. Check that the display preserved data

Report the maximum GDP per person in the 2007 table and the x-axis scale of the final figure. Did setting the scale replace GDP with its logarithm?

<details><summary>Solution 1</summary>

Run this complete solution after the worked example.

In [ ]:
print(year_2007["gdp_pc"].max())
print(ax_readable.get_xscale())

The table still stores GDP in dollars; the axis scale is log. Transforming a stored column requires a separate assignment, such as the one taught in 2.3.

</details>

### 2. Label Japan instead

Draw the same log-axis scatter and annotate Japan. Preserve the sample and units.

<details><summary>Solution 2</summary>

Run this complete solution after the worked example.

In [ ]:
#| label: fig-exercise-japan-label
#| fig-alt: "The 2007 log-income scatter with Japan annotated at life expectancy 82.603 years, the highest in this extract for that year."
#| fig-cap: "Historical Gapminder teaching extract via Jennifer Bryan (CC0). Country observations receive equal weight unless stated otherwise."
japan = year_2007.loc[year_2007["country"] == "Japan"].iloc[0]
fig_japan, ax_japan = plt.subplots(figsize=(6, 4.5), layout="constrained")
ax_japan.scatter(year_2007["gdp_pc"], year_2007["life_exp"], color="#21645a")
ax_japan.set_xscale("log")
ax_japan.set(xlabel="GDP per person (inflation-adjusted US dollars; log axis)", ylabel="Life expectancy (years)", title="Same sample · Japan annotated")
ax_japan.annotate("Japan", (japan["gdp_pc"], japan["life_exp"]), xytext=(-45, -24), textcoords="offset points", arrowprops={"arrowstyle": "-"})
plt.show()

The selected country changes, but the sample and scale do not. Annotation choices should serve the question rather than select only points that support a preferred story.

</details>

### 3. Write an honest caption

Print a caption stating the year, unit of observation, scale, weighting and limitation on causal interpretation.

<details><summary>Solution 3</summary>

Run this complete solution after the worked example.

In [ ]:
caption = "142 countries in 2007; one equally sized point per country. GDP per person uses a log axis in the historical extract. An association does not establish a causal income effect. Source: Gapminder via Jennifer Bryan, CC0."
print(caption)

This caption identifies the essential choices. A caption and alt text have different jobs: the caption documents scope, while alt text also describes the chart's key visual pattern for a reader who cannot see it.

</details>

## When it goes wrong

The following examples deliberately trigger specific errors. The prepared `try`/`except` wrapper catches that named error and prints its message so the notebook can continue. It is for these demonstrations; an unexpected error elsewhere should still stop execution. You can read the message without memorizing the wrapper.

### A scale name is not recognized

Use an intentionally invalid scale label.

In [ ]:
try:
    ax_readable.set_xscale("logarithmic_typo")
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

The supported name is "log". Invalid scale names should stop you before interpreting the plot.

### An annotation column is absent

A country row still uses the source column names.

In [ ]:
try:
    print(canada["income"])
except KeyError as error:
    print(f"{type(error).__name__}: {error}")

The field is gdp_pc. Verify the observation year and measurement definition as well as spelling; a pretty label cannot correct the wrong underlying variable.

## Recap

- Label units and year, and inspect excluded values before plotting.
- Log axes show ratios; changing an axis leaves stored values unchanged.
- Annotations should clarify rather than select a convenient story.
- Supply useful alt text and a scope/weighting caption; the pattern remains descriptive.

**Next:** [3.3 Comparing groups and small multiples](../03-viz/03-groups-small-multiples.ipynb).